# Intervention Threshold Pipeline

This section describes the pipeline used to evaluate **when** an intervention should be triggered based on *observed diagnoses*, and **how much** each intervention reduces the *true infections* in the simulated epidemic. 
The notebook below demonstrates how different trigger rules and intervention strategies can be combined and compared.

## Overview
The pipeline systematically tests multiple combinations of:
1. **Trigger types**  
2. **Trigger values**  
3. **Intervention packages**  
4. **Intervention durations**

For each combination, we run a full simulation and measure the resulting **total true infections** over the same time horizon.  
This allows us to identify which policy (trigger + intervention) is most effective.

All components are parameterized and can be adjusted.

## 1) Trigger Types

The pipeline supports several diagnosis‑based trigger rules:

### **• Absolute case threshold**
Trigger when the n days average of observed diagnoses exceeds a fixed value.  
Examples:  
- 7‑day avg ≥ 50  
- 7‑day avg ≥ 100  
- 7‑day avg ≥ 200  

### **• Weekly growth rate**
Trigger when cases grow rapidly compared to the previous week.  
Examples:  
- C̄(t) / C̄(t−7) ≥ 1.3  
- C̄(t) / C̄(t−7) ≥ 1.5  

### **• Slope‑based trigger**
Trigger when the daily increase exceeds a threshold.  
Examples:  
- dC/dt ≥ 10  
- dC/dt ≥ 20  

### **• Sustained increase**
Trigger when cases rise for several consecutive days.  
Examples:  
- 5 days of consecutive increases  
- 7 days of consecutive increases  

Each trigger type can take multiple parameter values, forming part of the grid search.

## 2) Intervention Packages

Each intervention package specifies how epidemic parameters change once the trigger fires.  
Examples include:

- **Mild distancing**  
  - β → β × 0.7  
- **Strong distancing**  
  - β → β × 0.4  
- **Enhanced testing/tracing**  
  - Increase test probability 

These packages are modular and can be extended.

## 3) Intervention Duration

Once triggered, an intervention can be enforced for:

- a **minimum number of days** (ex. 30, 45, 60)  
- **until cases fall below a release threshold**  
- a combination of both.

Duration is another dimension of the grid.

## 4) Grid Search Over All Combinations

The pipeline loops over:

- trigger types  
- trigger values  
- intervention packages  
- intervention durations  

For each combination, it:

1. Runs a full Covasim simulation  
2. Applies the observation model  
3. Checks when the trigger fires  
4. Activates the intervention  
5. Records outcomes

This produces a structured table of results for comparison.


## 5) Evaluation Using True Infections

Although triggers rely only on **observed diagnoses**,  
the evaluation uses **true infections** from the simulation:

- total infections  
- peak infections  
- timing of the peak  
- number of intervention days (policy cost)

The primary metric used in this notebook is:

### **Total true infections over the simulation period**

This allows us to identify which policy combination yields the greatest reduction in infections.

## Notebook Demonstration

The notebook below demonstrates:

- how to configure trigger rules  
- how to define intervention packages  
- how to run the grid search  
- how to compute outcomes  
- how to compare policies using tables and plots  

All parameters (trigger values, intervention strengths, durations, etc) are fully adjustable.

# Intervention Search Engine Modules

This notebook uses the simulation engine implemented in the
`intervention_search/` directory. The engine consists of:

- **runner.py** - day-by-day simulation orchestrator  
- **evaluator.py** - computes metrics for policy comparison  
- **triggers.py** - diagnosis-based trigger rules  
- **interventions.py** - intervention packages  
- **observation_model.py** - noisy surveillance model  
- **plot_result.py** - visualization utilities  
- **run_grid.py** - grid search over trigger × intervention combinations  

These modules allow us to test realistic policy strategies by combining:
1. noisy observed diagnoses,
2. trigger rules,
3. dynamic interventions,
4. resource constraints,
5. tradeoff scoring.

The notebook demonstrates how to use these components to evaluate
policy effectiveness under limited resources

In [2]:
import sys 
sys.path.append("../intervention_search")
import numpy as np
import pandas as pd
import covasim as cv

from runner import run_simulation
from evaluator import evaluate_results
from triggers import weekly_growth, relative_threshold
from interventions import reduce_beta, increase_testing

In [4]:
sim_pars = {
    "pop_size": 100000,
    "pop_infected": 50,
    "beta": 0.015,
    "n_days": 120,
}

obs_pars = {
    "p_test": 0.2,
    "p_seq": 0.1,
    "delay_pmf": [0.5, 0.3, 0.2],
}

dummy_sim = cv.Sim(sim_pars)

trigger_grid = [
    ("growth2", weekly_growth, {"ratio": 2, "window": 7}),
    ("growth5", weekly_growth, {"ratio": 5, "window": 7}),
    ("rel0.1", relative_threshold, {"proportion": 0.1, "window": 7, "sim": dummy_sim}),
    ("rel0.5", relative_threshold, {"proportion": 0.5, "window": 7, "sim": dummy_sim}),
]

intervention_grid = [
    ("beta70", lambda sim: reduce_beta(sim, 0.7)),
    ("beta50", lambda sim: reduce_beta(sim, 0.5)),
    ("test40", lambda sim: increase_testing(sim, 0.4)),
]


In [ ]:
results_list = []

for trig_name, trig_fn, trig_params in trigger_grid:
    for int_name, int_fn in intervention_grid:

        res = run_simulation(
            sim_pars=sim_pars,
            obs_pars=obs_pars,
            trigger_list=[(trig_name, trig_fn, trig_params)],
            intervention_fn=int_fn,
        )

        metrics = evaluate_results(res)

        results_list.append({
            "trigger": trig_name,
            "intervention": int_name,
            "results": res,
            "metrics": metrics,
        })


Initializing sim with 100000 people for 120 days
Initializing sim with 100000 people for 120 days
Initializing sim with 100000 people for 120 days
Initializing sim with 100000 people for 120 days
Initializing sim with 100000 people for 120 days
Initializing sim with 100000 people for 120 days
Initializing sim with 100000 people for 120 days
Initializing sim with 100000 people for 120 days
Initializing sim with 100000 people for 120 days
Initializing sim with 100000 people for 120 days
Initializing sim with 100000 people for 120 days
Initializing sim with 100000 people for 120 days


In [ ]:
rows = []
for r in results_list:
    m = r["metrics"]
    rows.append({
        "trigger": r["trigger"],
        "intervention": r["intervention"],
        "total_inf": m["total_infections"],
        "total_seq": m["total_sequences"],
        "num_triggers": len(r["results"]["trigger_events"]),
    })

df = pd.DataFrame(rows)
df

best_total_inf = min(results_list, key=lambda r: r["metrics"]["total_infections"])
best_total_deaths = min(results_list, key=lambda r: r["metrics"]["total_sequences"])
best_shortest_intervention = min(results_list, key=lambda r: len(r["results"]["trigger_events"]))

def summarize(run):
    return {
        "trigger": run["trigger"],
        "intervention": run["intervention"],
        "total_inf": run["metrics"]["total_infections"],
        "total_seq": run["metrics"]["total_sequences"],
        "num_triggers": len(run["results"]["trigger_events"]),
        "trigger_days": run["results"]["trigger_events"],
    }

summary_df = pd.DataFrame([
    summarize(best_total_inf),
    summarize(best_total_deaths),
    summarize(best_shortest_intervention),
], index=["best_total_inf", "best_total_deaths", "best_shortest_intervention"])

summary_df